In [27]:
import csv
import importlib
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

from time import sleep
from collections import deque
from itertools import count
from typing import Any, Dict, List, Optional, Tuple, Set

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.EnvWrapperLru import EnvWrapper

from RL.Adapters import FeatureAdapter, NetworkAdapter

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from '/home/eduardo/Workspace/CacheVideoPredict360/Sources/Common/utils.py'>

In [28]:
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey  # (vid, layer, tile, gop)

cfg = config.Config()

cfg.filename = f"lru.csv"
cfg.n_episodes = 100

debugger = debugger.debug

In [29]:
class LruPolicy(CachePolicy):

    def __init__(self, max_videos: int = 50, cfg: Any = None) -> None:
        self.cfg = cfg
        self.cur_size = 0
        self.video_capacity = max_videos

        # Video LRU: oldest video (by any access to its tiles) at index 0
        self.video_access_order: List[int] = []

        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def put(
        self, 
        key: int, 
        value: Any, 
        size: int
    ):
        new_video, _ = value

        if new_video in self.video_idx:
            self.video_access_order.remove(new_video)
            self.video_access_order.append(new_video)
            return []

        slot = self.video_idx.index(-1) if -1 in self.video_idx else None
        if slot == None:
            lru_video = self.video_access_order.pop(0)
            slot = self.video_idx.index(lru_video)

        self.video_idx[slot] = new_video
        self.tile_idx[slot] = [-1] * self.cfg.viewport
        
        self.video_access_order.append(new_video)

        self.cur_size = sum(1 for v in self.video_idx if v != -1)
        return []

    def get(self, key: CacheKey) -> Optional[Any]:
        raise NotImplementedError("LRU policy does not support manual retrieval of individual keys.")

    def contains(self, key: CacheKey) -> bool:
        raise NotImplementedError("LRU policy does not support manual checking of individual keys.")
    
    def keys(self):
        raise NotImplementedError("LRU policy does not support manual retrieval of keys.")
    
    def remove(self, key: CacheKey) -> bool:
        raise NotImplementedError("LRU policy does not support manual removal of individual keys.")
    
    def get_capacity(self) -> int:
        return self.cur_size
    
    def clear(self) -> None:
        self.video_access_order.clear()
        self.cur_size = 0
        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def stats(self) -> Dict[str, Any]:
        return {
            'current_size': self.cur_size,
            'capacity': self.cfg.cache_size,
            'num_items': len([v for v in self.video_idx if v != -1])
        }

In [30]:
class LruPolicyv2(datatypes.CachePolicy):
    """
    Anytime we need to evict, we evict the least recently used item(s) based on the hierarchy: video → GOP → tile. Eviction order:
        1. Evict entire videos (all GOPs and tiles) if video count exceeds max_videos.
        2. If the video itself is still present, evict entire GOPs (base + all enhancements) if GOP count for that video exceeds max_gops_per_video.
        3. If the GOP itself is still present, evict enhancement tiles (layer=1) in LRU order if tile count for that (vid, gop) exceeds max_tiles_per_gop. Base layer (layer=0) is never evicted in this step.
    """
    def __init__(self, max_videos: int = 50, max_gops_per_video: int = 30, max_tiles_per_gop: int = 4) -> None:
        self.max_videos = max_videos
        self.max_gops_per_video = max_gops_per_video
        self.max_tiles_per_gop = max_tiles_per_gop
        
        # Main storage: key -> (value, size)
        self._store: Dict[CacheKey, Tuple[Any, int]] = {}
        
        # ---------- LRU structures ----------
        # Video LRU (oldest video at index 0)
        self._video_lru: List[int] = []

        # GOP LRU per video: vid -> [gop_id oldest..newest]
        self._gop_lru: Dict[int, List[int]] = {}

        # Tile LRU per (vid, gop): vid -> { gop -> [tile oldest..newest] }
        self._tile_lru: Dict[int, Dict[int, List[int]]] = {}

        # ---------- Indexes for fast deletion ----------
        # Keys per video: vid -> set(keys)
        self._keys_by_video: Dict[int, Set[CacheKey]] = {}

        # Base key index: (vid, gop) -> base_key (or None if absent)
        self._base_index: Dict[Tuple[int, int], Optional[CacheKey]] = {}

        # Enhancement index: vid -> gop -> tile -> key
        self._enh_index: Dict[int, Dict[int, Dict[int, CacheKey]]] = {}

    # --------------------------------------------------------------------- #
    # Helpers: LRU management
    # --------------------------------------------------------------------- #

    def _touch_video(self, vid: int) -> None:
        """Move a video to MRU position."""
        if vid in self._video_lru:
            self._video_lru.remove(vid)
        self._video_lru.append(vid)

    def _touch_gop(self, vid: int, gop: int) -> None:
        """Move a GOP within a video to MRU position."""
        lst = self._gop_lru.setdefault(vid, [])
        if gop in lst:
            lst.remove(gop)
        lst.append(gop)

    def _touch_tile(self, vid: int, gop: int, tile: int) -> None:
        """Move a tile within (vid, gop) to MRU position."""
        per_vid = self._tile_lru.setdefault(vid, {})
        lst = per_vid.setdefault(gop, [])
        if tile in lst:
            lst.remove(tile)
        lst.append(tile)

    # --------------------------------------------------------------------- #
    # Helpers: Index maintenance
    # --------------------------------------------------------------------- #

    def _index_add(self, key: CacheKey, size: int) -> None:
        vid, layer, tile, gop = key
        self._store[key] = (self._store.get(key, (None, 0))[0], size) if key in self._store else (None, size)
        # ensure key presence in store has been set by caller before indexing
        self._keys_by_video.setdefault(vid, set()).add(key)

        if layer == 0:
            self._base_index[(vid, gop)] = key
        else:
            self._enh_index.setdefault(vid, {}).setdefault(gop, {})[tile] = key

    def _index_remove(self, key: CacheKey) -> None:
        if key not in self._store:
            return
        vid, layer, tile, gop = key

        # remove from store
        del self._store[key]

        # remove from per-video set
        vset = self._keys_by_video.get(vid)
        if vset:
            vset.discard(key)
            if not vset:
                # if no keys left for this video, drop video LRU entry as well
                self._keys_by_video.pop(vid, None)
                if vid in self._video_lru:
                    self._video_lru.remove(vid)
                self._gop_lru.pop(vid, None)
                self._tile_lru.pop(vid, None)
                self._enh_index.pop(vid, None)

        # update base/enh indexes
        if layer == 0:
            if self._base_index.get((vid, gop)) == key:
                self._base_index.pop((vid, gop), None)
            # also drop GOP LRU entry if GOP becomes empty later
        else:
            per_vid = self._enh_index.get(vid, {})
            per_gop = per_vid.get(gop, {})
            if per_gop.get(tile) == key:
                per_gop.pop(tile, None)
            if not per_gop and gop in per_vid:
                per_vid.pop(gop, None)

        # clean LRU nodes for GOP/tile if needed
        if vid in self._gop_lru and gop in self._gop_lru[vid]:
            # keep GOP in LRU only if ANY key remains for that GOP
            if not self._has_any_key_for_gop(vid, gop):
                self._gop_lru[vid].remove(gop)

        if vid in self._tile_lru and gop in self._tile_lru[vid]:
            tl = self._tile_lru[vid][gop]
            if tile in tl:
                tl.remove(tile)
            if not tl:
                self._tile_lru[vid].pop(gop, None)

    def _has_any_key_for_gop(self, vid: int, gop: int) -> bool:
        """Return True if base or any enhancement exists for (vid, gop)."""
        if self._base_index.get((vid, gop)) is not None:
            return True
        return bool(self._enh_index.get(vid, {}).get(gop, {}))

    # --------------------------------------------------------------------- #
    # Helpers: Eviction primitives
    # --------------------------------------------------------------------- #

    def _evict_entire_video(self, evict_vid: int, out: List[CacheKey]) -> None:
        """Evict all keys for a given video (and clean indexes)."""
        keys = list(self._keys_by_video.get(evict_vid, set()))
        for k in keys:
            self._index_remove(k)
            out.append(k)

    def _evict_oldest_video_if_needed(self, out: List[CacheKey]) -> None:
        if self.max_videos is not None and self.max_videos > 0:
            while len(self._keys_by_video) > self.max_videos:
                if not self._video_lru:
                    break
                evict_vid = self._video_lru[0]
                # If the oldest video was just touched but isn't first, ensure we pick
                # the true oldest:
                # (List is maintained; this is just a safety check.)
                self._evict_entire_video(evict_vid, out)

    def _evict_oldest_gop_if_needed(self, vid: int, out: List[CacheKey]) -> None:
        if self.max_gops_per_video is not None and self.max_gops_per_video > 0:
            glist = self._gop_lru.setdefault(vid, [])
            # Count GOPs by existence in either base/enh indexes
            def _count_gops(v: int) -> int:
                present = set()
                # From base index
                for (v2, g) in self._base_index.keys():
                    if v2 == v:
                        present.add(g)
                # From enhancement index
                for g in self._enh_index.get(v, {}).keys():
                    present.add(g)
                return len(present)

            while _count_gops(vid) > self.max_gops_per_video and glist:
                oldest_gop = glist[0]
                self._evict_gop(vid, oldest_gop, out)

    def _evict_gop(self, vid: int, gop: int, out: List[CacheKey]) -> None:
        """Evict base + all enhancements of (vid, gop)."""
        # Base
        bkey = self._base_index.pop((vid, gop), None)
        if bkey is not None and bkey in self._store:
            self._index_remove(bkey)
            out.append(bkey)

        # Enhancements
        enh_map = self._enh_index.get(vid, {}).pop(gop, {})
        for tkey in list(enh_map.values()):
            if tkey in self._store:
                self._index_remove(tkey)
                out.append(tkey)

        # Clean GOP/tile LRU structures
        if vid in self._gop_lru and gop in self._gop_lru[vid]:
            self._gop_lru[vid].remove(gop)
        if vid in self._tile_lru:
            self._tile_lru[vid].pop(gop, None)

    def _evict_oldest_tile_if_needed(self, vid: int, gop: int, out: List[CacheKey]) -> None:
        if self.max_tiles_per_gop is not None and self.max_tiles_per_gop > 0:
            tile_list = self._tile_lru.setdefault(vid, {}).setdefault(gop, [])
            while len(tile_list) > self.max_tiles_per_gop:
                oldest_tile = tile_list[0]
                # remove enhancement only
                tkey = self._enh_index.get(vid, {}).get(gop, {}).get(oldest_tile)
                if tkey is not None:
                    self._index_remove(tkey)
                    out.append(tkey)
                else:
                    # stale tile in LRU (shouldn't happen, but guard anyway)
                    tile_list.pop(0)

    # --------------------------------------------------------------------- #
    # Public API
    # --------------------------------------------------------------------- #

    def get(self, key: CacheKey) -> Optional[Any]:
        if key not in self._store:
            return None
        vid, layer, tile, gop = key
        # touch LRU structures
        self._touch_video(vid)
        self._touch_gop(vid, gop)
        if layer > 0:
            self._touch_tile(vid, gop, tile)
        return self._store[key][0]

    def put(self, key: CacheKey, value: Any, size: int) -> List[CacheKey]:
        """
        Insert or update a key, possibly evicting entries to satisfy the capacities.
        Returns the list of evicted keys.
        """
        evicted: List[CacheKey] = []
        vid, layer, tile, gop = key

        # If key already exists, just update value/recency (no eviction needed for same item).
        if key in self._store:
            self._store[key] = (value, size)

            if layer == 0:
                self._touch_video(vid)

            self._touch_gop(vid, gop)

            if layer == 1:
                self._touch_tile(vid, gop, tile)

            return evicted

        # Insert in main store first (value, size); index after LRU touches
        self._store[key] = (value, size)

        # Touch LRU levels
        if layer == 0:
            self._touch_video(vid)
            
        self._touch_gop(vid, gop)

        # Update indexes
        self._keys_by_video.setdefault(vid, set()).add(key)
        if layer == 0:
            self._base_index[(vid, gop)] = key
        else:
            self._enh_index.setdefault(vid, {}).setdefault(gop, {})[tile] = key
            self._touch_tile(vid, gop, tile)

        # Capacity enforcement

        # (1) Video capacity → evict entire videos (video-LRU)
        self._evict_oldest_video_if_needed(evicted)

        # If the video itself was evicted in step (1), nothing else to do
        if vid not in self._keys_by_video:
            return evicted

        # (2) GOP capacity per video → evict oldest GOP(s) for this video (base + enh)
        self._evict_oldest_gop_if_needed(vid, evicted)

        # If this GOP was evicted as a side effect, bail out
        if not self._has_any_key_for_gop(vid, gop):
            return evicted

        # (3) Tile capacity per (vid, gop) → evict oldest enhancement tiles in SAME (vid, gop)
        if layer == 1:
            self._evict_oldest_tile_if_needed(vid, gop, evicted)

        return evicted

    def contains(self, key: CacheKey) -> bool:
        return key in self._store

    def remove(self, key: CacheKey) -> bool:
        """Explicit removal. If removing a base, it also cascades to remove enhancements of that (vid, gop)."""
        if key not in self._store:
            return False
        vid, layer, tile, gop = key
        evicted: List[CacheKey] = []
        if layer == 0:
            self._evict_gop(vid, gop, evicted)  # base + all enh in that GOP
            return True
        else:
            self._index_remove(key)
            return True

    def clear(self) -> None:
        self._store.clear()
        self._video_lru.clear()
        self._gop_lru.clear()
        self._tile_lru.clear()
        self._keys_by_video.clear()
        self._base_index.clear()
        self._enh_index.clear()

    def get_capacity(self) -> int:
        """Aggregate 'size' across stored keys. (Count-based policy still supported.)"""
        return sum(sz for (_, sz) in self._store.values())

    def stats(self) -> Dict[str, Any]:
        """Diagnostics for debugging or analytics."""
        num_videos = len(self._keys_by_video)
        num_keys = len(self._store)

        # count gops per video from indexes
        gops_per_video = {
            vid: len(
                set(
                    [g for (v2, g) in self._base_index.keys() if v2 == vid]
                ).union(
                    set(self._enh_index.get(vid, {}).keys())
                )
            )
            for vid in self._keys_by_video.keys()
        }

        return {
            "num_items": num_keys,
            "num_videos": num_videos,
            "max_videos": self.max_videos,
            "max_gops_per_video": self.max_gops_per_video,
            "max_tiles_per_gop": self.max_tiles_per_gop,
            "gops_per_video": gops_per_video,
            "video_lru": list(self._video_lru),
        }

    def keys(self):
        return self._store.keys()


In [31]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    agent
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'epsilon',
            'lr'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': round(float(total_reward), 2),
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'lr': f"{agent.scheduler.get_last_lr()[0]:.10f}" if agent else None,
            'epsilon': round(float(agent.epsilon), 4) if agent else None
        })

def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, (enh_hits + base_hits), (enh_misses + base_misses), base_hits, base_misses

def build_latency_model(cfg):
    """Build and return the MultiDULatencyModel."""
    P = cfg.n_nodes
    max_U = cfg.n_users

    return MultiDULatencyModel(
        P=P,
        max_U=max_U,
        R_M_D=80e6,
        R_C_M=125e6,
        mu=2e7,
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float),
        rhoT_p=[0.2],
        lambda_p=[0.05],
        du_fixed_delay=0.001,
        mec_fixed_delay=0.005,
        cloud_fixed_delay=0.1
    )

def build_environment(cfg):
    """Construct the full multi-component environment wrapper."""
    du_caches = []

    policy=LruPolicy(
        cfg=cfg,
        max_videos=cfg.cache_size,
    )

    # MEC Cache Engine
    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity,
        policy=policy
    )

    # User request generator
    users_env = UserRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        arrival_rate=cfg.arrival_rate,
        zipf_alpha=cfg.zipf_alpha
    )

    # Latency Model
    latency_model = build_latency_model(cfg)
    
    # Wrapping all into the main training environment
    return EnvWrapper(
        cfg=cfg,
        n=cfg.n,
        m=cfg.m,
        n_layers=cfg.n_layers,
        users_env=users_env,
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=latency_model,
        theta=cfg.theta,
        lam=cfg.lam,
        max_steps=cfg.max_steps,
        prefetch_fn=lambda cache, action: cache.lru_live_prefetching(action),
        reward_fn=lambda env, reqs: env.compute_reward(reqs),
        debugger=debugger
    )

In [32]:
def run_episode(episode, env, net_adapter, cfg):
    """Run one full training episode."""
    _, info = net_adapter.reset()

    total_reward = 0.0
    cache_hits = cache_misses = 0
    base_hits = base_misses = 0

    env.warmup_phase(net_adapter)

    for step in count():

        req_state = info.get("user_request", None)
        
        _, reward, _, info = env.step(req_state, net_adapter)

        delta_r, hits, misses, bs_hits, bs_miss = update_metrics(info, reward)
        total_reward += delta_r
        cache_hits += hits
        cache_misses += misses
        base_hits += bs_hits
        base_misses += bs_miss

        if net_adapter.env_is_done():
            break

        debugger.log('cache_hits', hits)
        debugger.log('cache_misses', misses)

        # print(f"Episode {episode} | Step {step} | Reward: {reward:.2f} | Total Reward: {total_reward:.2f} | Hits: {cache_hits} | Misses: {cache_misses}")

    return total_reward, cache_hits, cache_misses, base_hits, base_misses

def train(cfg):
    print("\n--- Starting DRL Caching System ---")

    env = build_environment(cfg)

    feature_adapter = FeatureAdapter(cfg, env)
    net_adapter = NetworkAdapter(cfg, env, feature_adapter)

    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")
    debug_path = os.path.join(cfg.path_results, date_dir)
    os.makedirs(debug_path, exist_ok=True)

    for episode in range(cfg.n_episodes):

        total_reward, hits, misses, bs_hits, bs_miss = run_episode(
            episode, env, net_adapter, cfg
        )

        save_training_results(
            path_=cfg.path_results + "/" + date_dir,
            filename=cfg.filename,
            ep=episode,
            total_reward=total_reward,
            cache_hits=hits,
            cache_misses=misses,
            agent=None
        )

        debug_path = os.path.join(cfg.path_results, date_dir)
        os.makedirs(debug_path, exist_ok=True)

        debugger.save_results(filepath=f"{debug_path}/debug_ep{episode}")
        debugger.clear()

        print(
            f"--- Episode {episode} | "
            f"R: {total_reward:.2f} | "
            f"HR: {hits / (hits + misses + 1e-9):.2f} | "
            f"BHR: {bs_hits / (bs_hits + bs_miss + 1e-9):.2f} ---"
        )
        print("-" * 50)

if __name__ == "__main__":
    train(cfg)


--- Starting DRL Caching System ---
NetworkAdapter initialized with capacity: 50 videos, 4 tiles per video
--- Episode 0 | R: 0.00 | HR: 0.21 | BHR: 0.27 ---
--------------------------------------------------
--- Episode 1 | R: 0.00 | HR: 0.25 | BHR: 0.32 ---
--------------------------------------------------
--- Episode 2 | R: 0.00 | HR: 0.22 | BHR: 0.29 ---
--------------------------------------------------
--- Episode 3 | R: 0.00 | HR: 0.22 | BHR: 0.28 ---
--------------------------------------------------
--- Episode 4 | R: 0.00 | HR: 0.22 | BHR: 0.29 ---
--------------------------------------------------
--- Episode 5 | R: 0.00 | HR: 0.21 | BHR: 0.27 ---
--------------------------------------------------
--- Episode 6 | R: 0.00 | HR: 0.25 | BHR: 0.33 ---
--------------------------------------------------
--- Episode 7 | R: 0.00 | HR: 0.25 | BHR: 0.32 ---
--------------------------------------------------
--- Episode 8 | R: 0.00 | HR: 0.25 | BHR: 0.33 ---
-------------------------

In [33]:
# ----------------------------------------------------------------      
# 1. Publication Style Configuration
# ----------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True
})

# Professional color for line plots
PRIMARY_COLOR = "#2b7bba"

# ----------------------------------------------------------------
# 2. Data Loading & Smoothing
# ----------------------------------------------------------------
path = cfg.path_results + "/" + cfg.filename

print(f"Loading data from: {path}")

df = pd.read_csv(path)

total = (df["cache_hits"] + df["cache_misses"]).replace(0, np.nan)
df["hit_rate"] = (df["cache_hits"] / total) * 100
df["miss_rate"] = (df["cache_misses"] / total) * 100

# Metrics to plot
metrics = ["total_reward", "hit_rate", "miss_rate", "epsilon", "lr"]
window_size = 10  # Adjust smoothing window as needed

# ----------------------------------------------------------------
# 3. Plotting logic
# ----------------------------------------------------------------
# Adjusted figsize for a 4-column row (standard for full-width paper figures)
fig, axes = plt.subplots(1, len(metrics), figsize=(12, 3), sharex=True)

for ax, col in zip(axes, metrics):
    # Plot raw data with transparency (alpha)
    ax.plot(df["episode"], df[col], color=PRIMARY_COLOR, alpha=0.3, linewidth=0.8, label='Raw')
    
    # Plot moving average for clearer trend (except for epsilon which is usually linear)
    if col != "epsilon":
        smoothed = df[col].rolling(window=window_size).mean()
        ax.plot(df["episode"], smoothed, color=PRIMARY_COLOR, linewidth=1.5, label='Trend')
    else:
        # Just a solid line for Epsilon
        ax.plot(df["episode"], df[col], color=PRIMARY_COLOR, linewidth=1.5)

    # Stylistic cleanup
    ax.set_title(col.replace("_", " ").title(), fontweight="bold")
    ax.set_xlabel("Episode")
    
    # Remove redundant Y-labels to save space, or keep for clarity
    ax.set_ylabel("Value") 
    
    series = df[col].dropna()
    if not series.empty:
        ymin = series.min()
        ymax = series.max()
        pad = (ymax - ymin) * 0.05 if ymax != ymin else 1.0
        ax.set_ylim(ymin - pad, ymax + pad)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    # Tufte-style: remove top/right spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

# Optional: Add a single legend to the first plot if needed
# axes[0].legend(frameon=False)

plt.show()
fig.savefig("drl_caching_metrics.png", dpi=300)

Loading data from: /home/eduardo/Workspace/CacheVideoPredict360/Results/lru.csv


FileNotFoundError: [Errno 2] No such file or directory: '/home/eduardo/Workspace/CacheVideoPredict360/Results/lru.csv'